# Push-higher Kaggle run

Self-contained notebook that trains the hard-neg CLIP, rebuilds the gallery index, and evaluates C-HN α=0.7 with bootstrap. **Edit the config cell below to pick a variant**, then Run All.

Launch **3 parallel Kaggle kernels** with different configs to map the design space in ~2 hours wall-clock:

| Kernel | EPOCHS | AUGMENT | LAST_N_BLOCKS | Expected R@10 |
|---|---|---|---|---|
| A — current baseline | 10 | False | 4 | 0.881 (our reported) |
| **B — augmented (Recommended)** | 15 | **True** | 4 | 0.90–0.92 (target) |
| C — longer only | 20 | False | 4 | 0.89–0.91 |
| D — both | 20 | **True** | 4 | 0.92–0.94 (best guess) |

**Datasets to attach:** `ashok1145/vr-fproj`, `taralsanka/blip-updated-captions`, `taralsanka/best-yolo-pt` (only if using YOLO crops; we use bbox annotations).

**Settings:** GPU T4 ×1 (or P100), Internet OFF (no HF downloads needed).

In [1]:
# ════════════════════════════════════════════════════════════════════════════
# ── CONFIG — edit these for each parallel run ─────────────────────────────
# ════════════════════════════════════════════════════════════════════════════
# 4 preset variants (set SUFFIX accordingly so JSON outputs don't collide):
#
# | Variant         | EPOCHS | AUG  | BLOCKS | LR     | TRI_W | TRI_M | BS | ACCUM |
# |-----------------|--------|------|--------|--------|-------|-------|----|-------|
# | _aug15ep        |   15   | True |   4    | 1e-5   | 0.5   | 0.3   | 32 |   1   |   ← our augmented run
# | _20ep           |   20   | False|   4    | 1e-5   | 0.5   | 0.3   | 32 |   1   |   ← longer only
# | _aug20ep        |   20   | True |   4    | 1e-5   | 0.5   | 0.3   | 32 |   1   |   ← longer + aug
# | _others_setup   |    8   | True |   6    | 2e-5   | 0.35  | 0.25  | 16 |   1   |   ← match the 0.95-recall teammate

SEED            = 588
EPOCHS          = 15
AUGMENT         = True
LAST_N_BLOCKS   = 6
SUFFIX          = '_aug15ep'
REFRESH_EVERY   = 3

# Loss + optimisation
FT_LR           = 2e-5
FT_BATCH_SIZE   = 16
GRAD_ACCUM_STEPS = 1
TEMPERATURE     = 0.07
TRIPLET_MARGIN  = 0.25
LAMBDA_TRIPLET  = 0.35

# Retrieval
ALPHA           = 0.7
# ════════════════════════════════════════════════════════════════════════════

print(f'Variant: SEED={SEED}, EPOCHS={EPOCHS}, AUG={AUGMENT}, blocks={LAST_N_BLOCKS}, '
      f'lr={FT_LR}, λ_tri={LAMBDA_TRIPLET}, margin={TRIPLET_MARGIN}, suffix={SUFFIX!r}')

Variant: SEED=588, EPOCHS=15, AUG=True, blocks=6, lr=2e-05, λ_tri=0.35, margin=0.25, suffix='_aug15ep'


In [2]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install -q open-clip-torch hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00


In [3]:
import json, random
from collections import defaultdict
from pathlib import Path

import hnswlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
import open_clip

# Paths — Kaggle inputs
DATASET_ROOT  = Path('/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset')
IMG_ROOT      = DATASET_ROOT / 'img' / 'img'
SPLIT_FILE    = DATASET_ROOT / 'eval' / 'list_eval_partition.txt'
BBOX_FILE     = DATASET_ROOT / 'Anno' / 'list_bbox_inshop.txt'
CAPTIONS_FILE = Path('/kaggle/input/datasets/taralsanka/blip-updated-captions/captions.json')
OUTPUT_DIR    = Path('/kaggle/working')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_MODEL = 'ViT-L-14'
CLIP_PRETRAIN = 'openai'
EMB_DIM = 768
PAD = 0.05
HN_POOL_SIZE = 10
HN_EMB_BATCH = 64
FETCH_K = HN_POOL_SIZE * 5 + 1

print('device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
for p in [DATASET_ROOT, IMG_ROOT, SPLIT_FILE, BBOX_FILE, CAPTIONS_FILE]:
    print(f'  {str(p):<80} exists={p.exists()}')

device: cuda
GPU : Tesla T4
VRAM: 15.6 GB
  /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset                  exists=True
  /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset/img/img          exists=True
  /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset/eval/list_eval_partition.txt exists=True
  /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset/Anno/list_bbox_inshop.txt exists=True
  /kaggle/input/datasets/taralsanka/blip-updated-captions/captions.json            exists=True


In [4]:
# ── Helpers ──────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def parse_split(path):
    with open(path) as f: lines = f.readlines()
    rows = []
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 3: continue
        nm = parts[0][4:] if parts[0].startswith('img/') else parts[0]
        rows.append({'image_name': nm, 'item_id': parts[1], 'split': parts[2]})
    return rows

def parse_bbox(path):
    with open(path) as f: lines = f.readlines()
    bboxes = {}
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 7: continue
        nm = parts[0][4:] if parts[0].startswith('img/') else parts[0]
        bboxes[nm] = (int(parts[3]), int(parts[4]), int(parts[5]), int(parts[6]))
    return bboxes

def bbox_crop(pil, bbox, pad=PAD):
    W, H = pil.size
    x1, y1, x2, y2 = bbox
    px = int((x2-x1)*pad); py = int((y2-y1)*pad)
    x1=max(0,x1-px); y1=max(0,y1-py); x2=min(W,x2+px); y2=min(H,y2+py)
    return pil.crop((x1,y1,x2,y2)) if x2>x1 and y2>y1 else pil

all_rows = parse_split(SPLIT_FILE)
bbox_map = parse_bbox(BBOX_FILE)
captions = json.load(open(CAPTIONS_FILE))
train_rows   = [r for r in all_rows if r['split'] == 'train']
gallery_rows = [r for r in all_rows if r['split'] == 'gallery']
query_rows   = [r for r in all_rows if r['split'] == 'query']
print(f'Train: {len(train_rows)}  Gallery: {len(gallery_rows)}  Query: {len(query_rows)}')
print(f'Captions: {len(captions)}')

Train: 25882  Gallery: 12612  Query: 14218
Captions: 38494


In [5]:
# ── Mine hard negatives with frozen CLIP ─────────────────────────────────
clip_frozen, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAIN)
clip_frozen = clip_frozen.to(DEVICE).eval()

@torch.no_grad()
def embed_rows(rows, model, preprocess, batch_size=HN_EMB_BATCH):
    model.eval()
    all_embs, all_ids, all_names = [], [], []
    for i in tqdm(range(0, len(rows), batch_size), desc='Embedding', leave=False):
        batch = rows[i:i+batch_size]
        crops, ids, names = [], [], []
        for r in batch:
            p = IMG_ROOT / r['image_name']
            if not p.exists(): continue
            try:
                pil = Image.open(p).convert('RGB')
                bbox = bbox_map.get(r['image_name'])
                crop = bbox_crop(pil, bbox) if bbox else pil
                crops.append(preprocess(crop))
                ids.append(r['item_id'])
                names.append(r['image_name'])
            except Exception:
                continue
        if not crops: continue
        t = torch.stack(crops).to(DEVICE)
        emb = F.normalize(model.encode_image(t).float(), dim=-1)
        all_embs.append(emb.cpu().numpy())
        all_ids.extend(ids); all_names.extend(names)
    return np.vstack(all_embs).astype(np.float32), all_ids, all_names

def build_hn_pool(embs, ids, names):
    id_arr = np.array(ids)
    idx = hnswlib.Index(space='cosine', dim=EMB_DIM)
    idx.init_index(max_elements=len(embs), ef_construction=200, M=32)
    idx.add_items(embs, list(range(len(embs))))
    idx.set_ef(100)
    labels, _ = idx.knn_query(embs, k=FETCH_K)
    pool = {}
    for i, name in enumerate(names):
        hard_negs = []
        for pos in labels[i]:
            if pos == i or id_arr[pos] == ids[i]: continue
            hard_negs.append(names[pos])
            if len(hard_negs) == HN_POOL_SIZE: break
        pool[name] = hard_negs
    return pool

print('Embedding train images with frozen CLIP...')
train_embs, train_ids, train_names = embed_rows(train_rows, clip_frozen, clip_preprocess)
print(f'Embedded {len(train_names)}. Building hard-neg pool...')
hard_neg_pool = build_hn_pool(train_embs, train_ids, train_names)
print(f'Pool: avg {np.mean([len(v) for v in hard_neg_pool.values()]):.1f} neg/anchor')

del clip_frozen, train_embs
torch.cuda.empty_cache()

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Embedding train images with frozen CLIP...


Embedded 25882. Building hard-neg pool...
Pool: avg 10.0 neg/anchor


In [6]:
# ── Dataset with optional augmentation ───────────────────────────────────
# OpenCLIP default preprocessing: Resize(224) -> CenterCrop(224) -> ToTensor -> Normalize
# When AUGMENT=True we replace Resize+CenterCrop with RandomResizedCrop and add
# ColorJitter + HFlip BEFORE the normalisation, so it stays compatible.

OPENCLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
OPENCLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

if AUGMENT:
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(OPENCLIP_MEAN, OPENCLIP_STD),
    ])
    print('AUGMENT=True: using RandomResizedCrop + ColorJitter + HFlip on positives.')
else:
    train_transform = clip_preprocess  # the OpenCLIP default — deterministic
    print('AUGMENT=False: using OpenCLIP default preprocessing.')


class HardNegTripletDataset(Dataset):
    def __init__(self, rows, hard_neg_pool, transform):
        groups = defaultdict(list)
        for r in rows:
            groups[r['item_id']].append(r['image_name'])
        self.items = [(iid, imgs) for iid, imgs in groups.items()
                      if len(imgs) >= 2 and any(hard_neg_pool.get(n) for n in imgs)]
        self.hard_neg_pool = hard_neg_pool
        self.transform = transform
    def __len__(self): return len(self.items)
    def _load(self, name):
        pil = Image.open(IMG_ROOT / name).convert('RGB')
        bbox = bbox_map.get(name)
        crop = bbox_crop(pil, bbox) if bbox else pil
        return self.transform(crop)
    def __getitem__(self, idx):
        _, imgs = self.items[idx]
        anc, pos = random.sample(imgs, 2)
        hn_pool = self.hard_neg_pool.get(anc) or self.hard_neg_pool.get(pos) or []
        try:
            a = self._load(anc); p = self._load(pos)
            n = self._load(random.choice(hn_pool)) if hn_pool else a
        except Exception:
            a = p = n = self._load(anc)
        return a, p, n

hn_dataset = HardNegTripletDataset(train_rows, hard_neg_pool, train_transform)
print(f'Triplet dataset: {len(hn_dataset)} unique items')

AUGMENT=True: using RandomResizedCrop + ColorJitter + HFlip on positives.
Triplet dataset: 3985 unique items


In [7]:
# ── Train CLIP ───────────────────────────────────────────────────────────
set_seed(SEED)
clip_model, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAIN)
clip_model = clip_model.to(DEVICE)

for p in clip_model.parameters(): p.requires_grad = False
for block in list(clip_model.visual.transformer.resblocks)[-LAST_N_BLOCKS:]:
    for p in block.parameters(): p.requires_grad = True
for p in clip_model.visual.ln_post.parameters(): p.requires_grad = True
if hasattr(clip_model.visual, 'proj') and clip_model.visual.proj is not None:
    clip_model.visual.proj.requires_grad = True

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in clip_model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

def infonce_loss(z1, z2, temp=TEMPERATURE):
    B = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)
    sim = (z @ z.T) / temp
    sim.masked_fill_(torch.eye(2*B, dtype=torch.bool, device=z.device), float('-inf'))
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, labels)

triplet_loss_fn = nn.TripletMarginWithDistanceLoss(
    distance_function=lambda a, b: 1.0 - F.cosine_similarity(a, b),
    margin=TRIPLET_MARGIN, reduction='mean',
)

loader = DataLoader(hn_dataset, batch_size=FT_BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, clip_model.parameters()),
                              lr=FT_LR, weight_decay=0.01)
steps_per_epoch = max(1, len(loader) // GRAD_ACCUM_STEPS)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = 1 * steps_per_epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: step / max(1, warmup_steps) if step < warmup_steps
    else 0.5 * (1 + np.cos(np.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps)))
)
scaler = torch.amp.GradScaler('cuda')

def refresh_pool(model):
    print('  Refreshing hard-neg pool...')
    embs, ids, names = embed_rows(train_rows, model, clip_preprocess)
    hn_dataset.hard_neg_pool = build_hn_pool(embs, ids, names)
    del embs; torch.cuda.empty_cache()

clip_model.train()
best_loss = float('inf')
history = []
ckpt_path = OUTPUT_DIR / f'clip_finetuned_hn{SUFFIX}.pt'
hist_path = OUTPUT_DIR / f'training_history_hn{SUFFIX}.json'

for epoch in range(EPOCHS):
    if epoch > 0 and epoch % REFRESH_EVERY == 0:
        refresh_pool(clip_model); clip_model.train()

    tot_loss = tot_nce = tot_tri = 0.0
    optimizer.zero_grad()
    for step, (a, p, n) in enumerate(tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)):
        a = a.to(DEVICE, non_blocking=True)
        p = p.to(DEVICE, non_blocking=True)
        n = n.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            z_a = F.normalize(clip_model.encode_image(a).float(), dim=-1)
            z_p = F.normalize(clip_model.encode_image(p).float(), dim=-1)
            z_n = F.normalize(clip_model.encode_image(n).float(), dim=-1)
            loss_nce = infonce_loss(z_a, z_p)
            loss_tri = triplet_loss_fn(z_a, z_p, z_n)
            loss = (loss_nce + LAMBDA_TRIPLET * loss_tri) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, clip_model.parameters()), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()
        tot_loss += loss.item() * GRAD_ACCUM_STEPS
        tot_nce  += loss_nce.item()
        tot_tri  += loss_tri.item()

    n = len(loader)
    avg = tot_loss/n; nce = tot_nce/n; tri = tot_tri/n
    history.append({'epoch': epoch+1, 'loss': avg, 'infonce': nce, 'triplet': tri,
                    'lr': scheduler.get_last_lr()[0]})
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Total: {avg:.4f} | InfoNCE: {nce:.4f} | Triplet: {tri:.4f}')
    if avg < best_loss:
        best_loss = avg
        torch.save(clip_model.state_dict(), ckpt_path)
    json.dump(history, open(hist_path, 'w'), indent=2)

print(f'\nDone. Best loss: {best_loss:.4f}. Saved -> {ckpt_path}')

Trainable: 76,365,824 / 427,616,513 (17.9%)


Epoch  1/15 | Total: 1.0639 | InfoNCE: 0.9895 | Triplet: 0.2127


Epoch  2/15 | Total: 0.4490 | InfoNCE: 0.3991 | Triplet: 0.1428


Epoch  3/15 | Total: 0.3260 | InfoNCE: 0.2863 | Triplet: 0.1133
  Refreshing hard-neg pool...


Epoch  4/15 | Total: 0.3018 | InfoNCE: 0.2344 | Triplet: 0.1928


...

Epoch 11/15 | Total: 0.1381 | InfoNCE: 0.0883 | Triplet: 0.1422


Epoch 12/15 | Total: 0.1296 | InfoNCE: 0.0819 | Triplet: 0.1363
  Refreshing hard-neg pool...


Epoch 13/15 | Total: 0.1277 | InfoNCE: 0.0785 | Triplet: 0.1406


Epoch 14/15 | Total: 0.1141 | InfoNCE: 0.0669 | Triplet: 0.1350


Epoch 15/15 | Total: 0.1217 | InfoNCE: 0.0739 | Triplet: 0.1365

Done. Best loss: 0.1141. Saved -> /kaggle/working/clip_finetuned_hn_aug15ep.pt


In [8]:
# ── Rebuild α=0.7 gallery index using the new checkpoint ─────────────────
clip_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
clip_model.eval()
tokenizer = open_clip.get_tokenizer(CLIP_MODEL)

@torch.no_grad()
def encode_text_batch(texts):
    toks = tokenizer(texts).to(DEVICE)
    return F.normalize(clip_model.encode_text(toks).float(), dim=-1).cpu().numpy()

@torch.no_grad()
def gen_fused_embeddings(rows, captions, alpha, batch_size=64):
    embs, ids, names = [], [], []
    for i in tqdm(range(0, len(rows), batch_size), desc=f'Gallery α={alpha}', leave=False):
        batch = rows[i:i+batch_size]
        crops, txts, b_ids, b_names = [], [], [], []
        for r in batch:
            p = IMG_ROOT / r['image_name']
            if not p.exists(): continue
            try:
                pil = Image.open(p).convert('RGB')
                bbox = bbox_map.get(r['image_name'])
                crop = bbox_crop(pil, bbox) if bbox else pil
                crops.append(clip_preprocess(crop))
                txts.append(captions.get(r['image_name'], '') or '')
                b_ids.append(r['item_id']); b_names.append(r['image_name'])
            except Exception:
                continue
        if not crops: continue
        t = torch.stack(crops).to(DEVICE)
        img_emb = F.normalize(clip_model.encode_image(t).float(), dim=-1).cpu().numpy()
        txt_emb = encode_text_batch(txts) if alpha < 1.0 else np.zeros_like(img_emb)
        fused = alpha * img_emb + (1 - alpha) * txt_emb
        fused = fused / (np.linalg.norm(fused, axis=1, keepdims=True) + 1e-9)
        embs.append(fused.astype(np.float32)); ids.extend(b_ids); names.extend(b_names)
    return np.vstack(embs), ids, names

g_embs, g_ids, g_names = gen_fused_embeddings(gallery_rows, captions, ALPHA)
idx = hnswlib.Index(space='cosine', dim=EMB_DIM)
idx.init_index(max_elements=len(g_embs), ef_construction=200, M=32)
idx.add_items(g_embs, list(range(len(g_embs))))
idx.set_ef(100)
bin_path = OUTPUT_DIR / f'gallery_index_C_alpha07_hn{SUFFIX}.bin'
meta_path = OUTPUT_DIR / f'gallery_meta_C_alpha07_hn{SUFFIX}.json'
idx.save_index(str(bin_path))
json.dump({'item_ids': g_ids, 'img_names': g_names}, open(meta_path, 'w'))
print(f'Gallery index: {bin_path.name} ({g_embs.shape})')

Gallery index: gallery_index_C_alpha07_hn_aug15ep.bin ((12612, 768))


In [9]:
# ── Evaluate C-HN α=0.7 on the full query set ────────────────────────────
@torch.no_grad()
def encode_queries(rows, batch_size=64):
    embs, ids = [], []
    for i in tqdm(range(0, len(rows), batch_size), desc='Encoding queries', leave=False):
        batch = rows[i:i+batch_size]
        crops, b_ids = [], []
        for r in batch:
            p = IMG_ROOT / r['image_name']
            if not p.exists(): continue
            try:
                pil = Image.open(p).convert('RGB')
                bbox = bbox_map.get(r['image_name'])
                crop = bbox_crop(pil, bbox) if bbox else pil
                crops.append(clip_preprocess(crop)); b_ids.append(r['item_id'])
            except Exception:
                continue
        if not crops: continue
        t = torch.stack(crops).to(DEVICE)
        e = F.normalize(clip_model.encode_image(t).float(), dim=-1).cpu().numpy()
        embs.append(e.astype(np.float32)); ids.extend(b_ids)
    return np.vstack(embs), ids

q_embs, q_ids = encode_queries(query_rows)
print(f'Query embeddings: {q_embs.shape}')

K_LIST = [5, 10, 15]
max_k = max(K_LIST)
labels, _ = idx.knn_query(q_embs, k=max_k)
g_id_arr = np.array(g_ids)

g_count = defaultdict(int)
for iid in g_ids: g_count[iid] += 1

def recall_hit(retrieved, qid, k):
    return int(any(i == qid for i in retrieved[:k]))
def recall_full(retrieved, qid, n_rel, k):
    if n_rel <= 0: return 0.0
    return sum(1 for i in retrieved[:k] if i == qid) / n_rel
def ndcg(retrieved, qid, n_rel, k):
    dcg = sum(1.0/np.log2(r+2) for r, i in enumerate(retrieved[:k]) if i == qid)
    idcg = sum(1.0/np.log2(r+2) for r in range(min(n_rel, k)))
    return dcg/idcg if idcg > 0 else 0.0
def ap(retrieved, qid, n_rel, k):
    hits, ps = 0, 0.0
    for r, i in enumerate(retrieved[:k], 1):
        if i == qid: hits += 1; ps += hits/r
    denom = min(n_rel, k)
    return ps/denom if denom > 0 else 0.0

per_q = {f'{m}@{k}': [] for m in ['Recall', 'Recall_full', 'NDCG', 'mAP'] for k in K_LIST}
for j, qid in enumerate(q_ids):
    ret = g_id_arr[labels[j]].tolist()
    nrel = g_count.get(qid, 0)
    for k in K_LIST:
        per_q[f'Recall@{k}'].append(recall_hit(ret, qid, k))
        per_q[f'Recall_full@{k}'].append(recall_full(ret, qid, nrel, k))
        per_q[f'NDCG@{k}'].append(ndcg(ret, qid, nrel, k))
        per_q[f'mAP@{k}'].append(ap(ret, qid, nrel, k))

results = {m: float(np.mean(v)) for m, v in per_q.items()}
# Bootstrap mean ± std over 4 query-set resamples
boot = {}
for m, scores in per_q.items():
    arr = np.asarray(scores)
    means_b = []
    for s in [83, 588, 527, 33]:
        rng = np.random.default_rng(s)
        idx_b = rng.integers(0, len(arr), size=int(len(arr)*0.8))
        means_b.append(float(arr[idx_b].mean()))
    boot[m] = {'mean': float(np.mean(means_b)), 'std': float(np.std(means_b, ddof=1))}

print('\n' + '='*55)
print(f'Results — C_alpha0.7_hn{SUFFIX}  (n={len(q_ids)})')
print('='*55)
for k in K_LIST:
    for m in ['Recall', 'Recall_full', 'NDCG', 'mAP']:
        mk = f'{m}@{k}'
        print(f'  {mk:<14} {results[mk]:.4f}   (boot {boot[mk]["mean"]:.4f} ± {boot[mk]["std"]:.4f})')

# ── Comparison with the 4-seed headline (baseline) ────────────────────────
B_R10, B_NDCG10, B_MAP10 = 0.881, 0.578, 0.480
print(f'\nBaseline (4-seed headline, last-4 blocks, 10 ep, no aug):')
print(f'  R@10={B_R10:.3f}  NDCG@10={B_NDCG10:.3f}  mAP@10={B_MAP10:.3f}')
print(f'This run ({SUFFIX!r}):')
print(f'  R@10={results["Recall@10"]:.3f}  NDCG@10={results["NDCG@10"]:.3f}  mAP@10={results["mAP@10"]:.3f}')
dR  = results["Recall@10"] - B_R10
dN  = results["NDCG@10"]   - B_NDCG10
dM  = results["mAP@10"]    - B_MAP10
print(f'Δ vs headline:')
print(f'  R@10={dR:+.3f} ({dR*100:+.1f} pp)  NDCG@10={dN:+.3f} ({dN*100:+.1f} pp)  mAP@10={dM:+.3f} ({dM*100:+.1f} pp)')

out = OUTPUT_DIR / f'eval_C_alpha0.7_hn{SUFFIX}.json'
json.dump({
    'condition': f'C_alpha0.7_hn{SUFFIX}',
    'alpha': ALPHA, 'epochs': EPOCHS, 'augment': AUGMENT, 'last_n_blocks': LAST_N_BLOCKS,
    'seed': SEED, 'n_queries': len(q_ids),
    'results': results, 'bootstrap': boot,
}, open(out, 'w'), indent=2)
print(f'\nSaved -> {out}')
print('\nDownload from the Kaggle Output tab:')
print(f'  {ckpt_path}\n  {bin_path}\n  {meta_path}\n  {hist_path}\n  {out}')

Query embeddings: (14218, 768)

Results — C_alpha0.7_hn_aug15ep  (n=14218)
  Recall@5       0.8756   (boot 0.8773 ± 0.0033)
  Recall_full@5  0.5506   (boot 0.5497 ± 0.0013)
  NDCG@5         0.6375   (boot 0.6378 ± 0.0023)
  mAP@5          0.5619   (boot 0.5618 ± 0.0023)
  Recall@10      0.9093   (boot 0.9101 ± 0.0031)
  Recall_full@10 0.6346   (boot 0.6338 ± 0.0015)
  NDCG@10        0.6464   (boot 0.6466 ± 0.0018)
  mAP@10         0.5541   (boot 0.5540 ± 0.0017)
  Recall@15      0.9274   (boot 0.9278 ± 0.0032)
  Recall_full@15 0.6799   (boot 0.6790 ± 0.0007)
  NDCG@15        0.6592   (boot 0.6593 ± 0.0014)
  mAP@15         0.5585   (boot 0.5583 ± 0.0015)

Baseline (4-seed headline, last-4 blocks, 10 ep, no aug):
  R@10=0.881  NDCG@10=0.578  mAP@10=0.480
This run ('_aug15ep'):
  R@10=0.909  NDCG@10=0.646  mAP@10=0.554
Δ vs headline:
  R@10=+0.028 (+2.8 pp)  NDCG@10=+0.068 (+6.8 pp)  mAP@10=+0.074 (+7.4 pp)

Saved -> /kaggle/working/eval_C_alpha0.7_hn_aug15ep.json

Download from the Kagg